## Import Library

In [7]:
%load_ext autoreload
%autoreload 2
from machine_lib import * 
from mylib import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1, Login
在machine_lib文件的login方法中填写用户名和密码后保存然后来到本文件Restart Kernal后重新import machine_lib

In [ ]:
s = login()

In [ ]:
df = get_datafields(s, dataset_id = 'analyst7', region='USA', universe='TOP3000', delay=1)

print(df)

## 2, get data fields

In [ ]:
df = get_datafields(s, dataset_id = 'pv87', region='USA', universe='ILLIQUID_MINVOL1M', delay=1)

print(df)

### Matrix Data

In [ ]:
#组装服务器读取字段
# print(df[df['type'] == "MATRIX"]["id"].tolist())
pc_fields = process_datafields(df, "matrix")
#组装本地字段
df1=['anl11_knrtcesger1pme/anl11_knrtcesger3pme','anl11_knrtcesger1pme/anl11_knrtcesgergse']
# pc_fields = process_datafieldsfromlocal(df1)
#组装模板字段
model='A( filter( sigmoid( if_else( greater(ts_zscore(B, 30), 1), ts_zscore(B, 30),0)),h="1 2 3 4", t="0.5" ),market)'
# A=['group_count','group_max','group_median','group_min','group_neutralize','group_rank','group_scale','group_std_dev','group_sum','group_zscore']
A=['group_rank']
# B=['news_max_dn_ret','news_max_up_amt']
C = ["MARKET", "INDUSTRY", 'SECTOR', 'SUBINDUSTRY', 'COUNTRY']
# B=loadtxt()
B=pc_fields
pc_fields=process_datafieldsfrommodel(A,B,C,model)
print('A:'+str(len(A))+',b:'+str(len(B))+',rs:'+str(len(pc_fields)))
printlist(pc_fields)

### Vector Data

In [ ]:
#df = get_datafields(s, dataset_id='news85', region='USA', universe='TOP3000', delay=1)
print(df[df['type'] == "VECTOR"]["id"].tolist())
print(process_datafields(df, "vector"))

## 3, Alpha factory
### start with First Order

In [ ]:
first_order = first_order_factory(pc_fields, ts_ops)
first_order=pc_fields
printlist(first_order)
print(len(first_order))

In [ ]:
# Pad initial decay with alpha
init_decay = 6
fo_alpha_list = []
for alpha in first_order:
    fo_alpha_list.append((alpha, init_decay))
print(len(fo_alpha_list))
printlist(fo_alpha_list)

In [ ]:
# Load alphas to task pools
pools = load_task_pool(fo_alpha_list, 10, 10)
print(pools[0])

## 4, simulate alphas

In [ ]:
# Simulate First Order
multi_simulate(pools, "MARKET", "USA", "ILLIQUID_MINVOL1M", 46)
#print to csv

## 5, Select alphas
go to web alphas penal to look for the number and date to track for next order improve

In [ ]:
## get promising alphas to improve in the next order
fo_tracker = get_alphas("11-06", "11-12", 1.2, 0.5, "USA", 320, "track")

## 6, Next order improvement - Second Order
second order: ts_ops(field, days) -> group_ops(ts_ops(field, days), group)

In [ ]:
print(len(fo_tracker['next']))
print(len(fo_tracker['decay']))

fo_layer = prune(fo_tracker['next'] + fo_tracker['decay'], 'usa', 'anl4', 5)
so_alpha_list = []
group_ops = group_ops = ["group_neutralize", "group_rank", "group_normalize", "group_scale", "group_zscore"]
for region, couples in fo_layer.items():
    for expr, decay in couples:
        for alpha in get_group_second_order_factory([expr], group_ops, region):
            so_alpha_list.append((alpha,decay))

print(len(so_alpha_list))
print(so_alpha_list[:3])

### Simulate second order

In [ ]:
so_pools = load_task_pool(so_alpha_list, 9, 9)
multi_simulate(so_pools, 'SUBINDUSTRY', 'USA', 'TOP3000', 0)

## Higher Order for improvement - Third Order
group_ops(ts_ops(field, days), group) -> trade_when(entre_event, group_ops(ts_ops(field, days), group), exit_event)

In [ ]:
## get promising alphas from second order to improve in the third order
so_tracker = get_alphas("10-10", "11-03", 0.5, 0.1, "USA", 100, "track")

print(len(so_tracker['next']))
print(len(so_tracker['decay']))

so_layer = prune(so_tracker['next'] + so_tracker['decay'], 'chn', 'mdl26', 5)
th_alpha_list = []
for region, couples in so_layer.items():
    for expr, decay in couples:
        for alpha in trade_when_factory("trade_when",expr,region):
            th_alpha_list.append((alpha,decay))
print(len(th_alpha_list))

### Simulate Third Order

In [ ]:
# Simulate third order
th_pools = load_task_pool(th_alpha_list, 9, 10)
multi_simulate(th_pools, 'SUBINDUSTRY', 'USA', 'TOP3000', 0)

## 7, Get submittable alphas


In [ ]:
# 1.58 sharpe, 1 fitness, "submit"参数
th_tracker = get_alphas("10-15", "11-20", 1.58, 1, "USA", 200, "submit")
print(th_tracker)

In [ ]:
## 将get的alpha的id取出至stone_bag，用api check submission
stone_bag = []
for alpha in th_tracker['next'] + th_tracker['decay']:
    stone_bag.append(alpha[0])
print(len(stone_bag))
gold_bag = []
check_submission(stone_bag, gold_bag, 0)

In [ ]:
# 打印可提交的alpha信息并按sharpe排序，在网页上找到alpha手动提交
view_alphas(gold_bag)
printlist(gold_bag)

## 8, fine-tune submittable alphas
neutralization, performance comparison, turnover, margin